# Restaurant Sales Analysis Project
### NoteBook 1: Q1_Data_Cleaning

## 1. Import Libraries


In [1]:
import pandas as pd

## 2. Load Data


In [2]:
df1 = pd.read_excel("../../../Data/raw/Q1.xlsx")

## 3. Initial Data Inspection

This section presents an initial assessment of the dataset, including its structure, data types, missing values, and records with zero values in key financial fields to identify potential data quality issues before the cleaning process.

### 3.1 Verify Dataset Structure

In [3]:
df1.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42375 entries, 0 to 42374
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    15835 non-null  datetime64[ns]
 1   Receipt Number          15835 non-null  object        
 2   Customer                6717 non-null   object        
 3   Invoice                 3590 non-null   object        
 4   Is Refunded             15835 non-null  float64       
 5   Order Lines/Product     42372 non-null  object        
 6   Order Lines/Quantity    42372 non-null  float64       
 7   Order Lines/Unit Price  42372 non-null  float64       
 8   Order Lines/Subtotal    42372 non-null  float64       
 9   Total                   15835 non-null  float64       
dtypes: datetime64[ns](1), float64(5), object(4)
memory usage: 3.2+ MB


### 3.2 Inspect Missing Values

In [4]:
df1.isna().sum()

Date                      26540
Receipt Number            26540
Customer                  35658
Invoice                   38785
Is Refunded               26540
Order Lines/Product           3
Order Lines/Quantity          3
Order Lines/Unit Price        3
Order Lines/Subtotal          3
Total                     26540
dtype: int64

### 3.3 Inspect transaction records for zero values in key financial fields.

In [5]:
# Count transaction records containing zero values in key financial fields.
zero_value_records = df1[
    (df1["Order Lines/Quantity"] == 0) |
    (df1["Order Lines/Subtotal"] == 0) |
    (df1["Total"] == 0) |
    (df1["Order Lines/Unit Price"] == 0)
]

print(f"Number of records requiring review: {len(zero_value_records)}")

Number of records requiring review: 483


### 3.4 Check for receipt numbers linked to multiple transaction dates.

In [6]:
df1.groupby("Receipt Number")["Date"].nunique().sort_values(ascending=False).head(8)

Receipt Number
طلب 05259-001-1120    2
طلب 04803-001-1716    2
طلب 05033-002-5466    2
طلب 04822-001-3897    2
طلب 05129-001-1532    2
طلب 04966-002-8350    2
طلب 04773-001-2496    2
طلب 04636-001-0001    1
Name: Date, dtype: int64

## 4. Rename Columns

Column names were renamed to improve readability and simplify the analysis.

In [7]:
df1.rename(columns={
    "Date": "Date",
    "Receipt Number": "Receipt Number",
    "Customer": "Customer",
    "Invoice": "Invoice",
    "Is Refunded": "Is Refunded", 
    "Order Lines/Product": "Product",
    "Order Lines/Quantity": "Quantity",
    "Order Lines/Unit Price": "Unit_Price",
    "Order Lines/Subtotal": "Subtotal",
    "Total": "Total"
}, inplace=True)

## 5. Data Cleaning, Preprocessing, and Data Consistency Checks

This section performs the main data cleaning and preprocessing steps to improve the quality and consistency of the dataset. First, a copy of the original dataset is created to preserve the raw data. Unnecessary columns (`Invoice` and `Is Refunded`) are removed as they are not required for the analysis. Records with missing product names are deleted following a manual inspection, since product information is essential for transaction analysis.

Next, forward filling is applied only to selected columns (`Date`, `Receipt Number`, `Quantity`, `Unit_Price`, and `Subtotal`) because these values are expected to remain consistent within the same transaction. The `Customer` column is intentionally excluded to avoid assigning incorrect customer names to unrelated transactions, while the `Total` column is excluded because it represents the overall transaction amount rather than individual product lines, and propagating its values could introduce inaccurate financial information.

A data consistency check is then performed to identify receipt numbers associated with multiple transaction dates. The inspection revealed a small number of duplicate receipt numbers linked to different transaction times. These duplicate cases were resolved by removing the oldest transaction record for each affected receipt number while retaining the most recent one. After applying this correction, the consistency check confirmed that only one receipt number differed from the original dataset, which was expected because an older duplicate transaction containing an invalid `Subtotal` value of zero had been removed. Finally, the dataset was inspected for fully duplicated records to ensure data integrity. Any exact duplicate records were identified and removed to eliminate redundant transaction entries. A final validation check was then performed to confirm that no duplicate records remained before proceeding to the exploratory data analysis.

In [8]:
# Create a copy of the original dataset to preserve the raw data.
df1_copy = df1.copy()
df1_copy = df1_copy.drop(columns=['Invoice', 'Is Refunded'])
df1_copy = df1_copy.dropna(subset=['Product'])

In [9]:
# Propagate transaction-level values to product rows within the same transaction.
columns_to_fill = [
    "Date",
    "Receipt Number",
    "Quantity",
    "Unit_Price",
    "Subtotal",

]


df1_copy[columns_to_fill] = df1_copy[columns_to_fill].ffill()

In [10]:
df1_copy = df1_copy[df1_copy['Subtotal'] != 0]

In [11]:
df1_copy.isna().sum()

Date                  0
Receipt Number        0
Customer          35595
Product               0
Quantity              0
Unit_Price            0
Subtotal              0
Total             26486
dtype: int64

In [12]:
# Check for receipt numbers linked to multiple transaction dates.
df1_copy.groupby("Receipt Number")["Date"].nunique().sort_values(ascending=False).head(8)

Receipt Number
طلب 04966-002-8350    2
طلب 04803-001-1716    2
طلب 05129-001-1532    2
طلب 05259-001-1120    2
طلب 04822-001-3897    2
طلب 05033-002-5466    2
طلب 05039-014-0002    1
طلب 05042-001-0001    1
Name: Date, dtype: int64

In [13]:
# Function Remove the oldest transaction (based on Date) for a given Receipt Number.
def remove_oldest_receipt_record(df, receipt_number):

    # Check if the receipt number exists
    if receipt_number not in df["Receipt Number"].values:
        print(f"Receipt Number '{receipt_number}' not found.")
        return df

    # Find the oldest date for the receipt
    old_date = df.loc[
        df["Receipt Number"] == receipt_number,
        "Date"
    ].min()

    # Remove all rows belonging to the oldest transaction
    df = df[
        ~(
            (df["Receipt Number"] == receipt_number) &
            (df["Date"] == old_date)
        )
    ]

    return df

In [14]:
df1_copy = remove_oldest_receipt_record(df1_copy,"طلب 05033-002-5466")
df1_copy = remove_oldest_receipt_record(df1_copy,"طلب 04966-002-8350")
df1_copy = remove_oldest_receipt_record(df1_copy,"طلب 04803-001-1716")
df1_copy = remove_oldest_receipt_record(df1_copy,"طلب 05129-001-1532")
df1_copy = remove_oldest_receipt_record(df1_copy,"طلب 05259-001-1120")
df1_copy = remove_oldest_receipt_record(df1_copy,"طلب 04822-001-3897")

In [15]:
# Check for duplicate records.
print(f"Duplicate records: {df1_copy.duplicated().sum()}")

Duplicate records: 32


In [16]:
# Remove duplicate records.
df1_copy = df1_copy.drop_duplicates()

In [17]:
# Verify that no duplicate records remain.
print(f"Duplicate records after removal: {df1_copy.duplicated().sum()}")

Duplicate records after removal: 0


## 6. Export Clean Dataset

In [18]:
df1_copy.to_excel("../../../Data/Cleaned/clean_q1_data.xlsx", index=False)